## **Multiclass classification data-preprocessing**

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

from config import (
    CSV_PATH,
    XRAY_DIR,
    OUT_DIR,
)

def load_data(csv_path):
    df = pd.read_csv(csv_path)
    rename_map = {
        "Image Index": "IMGPATH",
        "Finding Labels": "DISEASELABEL",
        "Follow-up #": "FOLLOWUP",
        "Patient ID": "PATID",
        "Patient Age": "AGE",
        "Patient Gender": "GENDER",
        "View Position": "VP",
    }
    df = df.rename(columns=rename_map)
    return df

def find_full_path(image_name):
    for subdir in os.listdir(XRAY_DIR):
        full_path = os.path.join(XRAY_DIR, subdir, image_name)
        if os.path.exists(full_path):
            return full_path
    return None

def filter_selected_classes(df, selected_classes):
    df["DISEASELABEL"] = df["DISEASELABEL"].apply(lambda x: x.split("|") if isinstance(x, str) else [x])
    df = df[df["DISEASELABEL"].apply(lambda labels: any(label in selected_classes for label in labels))].copy()
    return df

def check_multilabel_overlap(df, selected_classes):
    df = df[df["DISEASELABEL"].apply(lambda labels: sum(1 for label in labels if label in selected_classes) == 1)].copy()
    return df

def remove_random_percentage_by_class(df, keep_percentages):
    filtered_df = []
    for class_label, keep_percentage in keep_percentages.items():
        class_df = df[df["CLASS"] == class_label]
        random_keep = class_df.sample(frac=keep_percentage, random_state=42)
        filtered_df.append(random_keep)
    filtered_df = pd.concat(filtered_df)
    return filtered_df

def assign_class_labels(df, class_map):
    df["CLASS"] = df["DISEASELABEL"].apply(
        lambda labels: max((class_map.get(label, -1) for label in labels), default=-1)
    )
    df = df[
        [
            "IMGPATH",
            "DISEASELABEL",
            "CLASS",
            "FOLLOWUP",
            "PATID",
            "AGE",
            "GENDER",
            "VP",
        ]
    ]
    return df

def patient_aware_split(df, test_size=0.10, val_size=0.15):
    unique_patients = df["PATID"].unique()
    train_val_patients, test_patients = train_test_split(
        unique_patients, test_size=test_size, random_state=42
    )
    
    train_patients, val_patients = train_test_split(
        train_val_patients, test_size=val_size / (1 - test_size), random_state=42
    )
    
    train_df = df[df["PATID"].isin(train_patients)]
    val_df = df[df["PATID"].isin(val_patients)]
    test_df = df[df["PATID"].isin(test_patients)]
    
    return train_df, val_df, test_df

def preprocess_data(csv_path):
    selected_classes = {"No Finding", "Pneumothorax", "Effusion"}
    class_map = {"No Finding": 0, "Pneumothorax": 1, "Effusion": 2}
    
    df = load_data(csv_path)
    df["IMGPATH"] = df["IMGPATH"].apply(find_full_path)
    df = filter_selected_classes(df, selected_classes)
    df = check_multilabel_overlap(df, selected_classes)
    df = assign_class_labels(df, class_map)
    
    # Example: Keep 70% of class 1 (Pneumothorax) and 80% of class 2 (Effusion)
    df = remove_random_percentage_by_class(df, {0: 0.08, 1: 1, 2: 0.4})
    
    train_df, val_df, test_df = patient_aware_split(df)
    
    return train_df, val_df, test_df

train_df, val_df, test_df = preprocess_data(CSV_PATH)

train_df.to_csv(OUT_DIR / "train.csv", index=False)
val_df.to_csv(OUT_DIR / "val.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)
